# Agricultural Extension RAG: Student Starter Notebook

Welcome! In this competition you will build the **retrieval** part of a Retrieval-Augmented Generation (RAG) system. Given a farmer's query, your system must rank the five most useful agricultural documents.

This notebook is written for beginners. It provides:

1. a complete TF-IDF baseline that works without a GPU or internet access;
2. a local implementation of the competition metric, nDCG@5;
3. strict checks for a correctly formatted `submission.csv`;
4. an optional dense-retrieval example using a model attached through **Kaggle Models**.

> **Submission rule:** You must use a Kaggle Notebook. Run it from top to bottom, choose **Save Version → Save & Run All**, and submit `/kaggle/working/submission.csv` from the saved version's **Output** panel.

## 0. Kaggle notebook workflow

| Step | What to do |
|------|------------|
| 1 | Join the competition → **Code → New Notebook** (or open the official starter). |
| 2 | Confirm competition **data** is attached in the **Input** panel. |
| 3 | For dense retrieval: **Add Input → Models** (optional; CPU baseline needs no GPU). |
| 4 | Run all cells, then **Save Version → Save & Run All**. |
| 5 | Submit `submission.csv` from the saved version's **Output** panel. |

Keep the **committed Kaggle notebook URL** if organizers ask. Do not submit a CSV created only on your laptop without a matching saved Kaggle run.

## 1. Understand the task

- A **document corpus** is the collection of passages that can be retrieved.
- A **query** is a question or information need.
- **Qrels** (query relevance judgements) say how relevant a document is to a training query. Here relevance is graded from 0 to 3; larger is better.
- A **retriever** gives each document a score for a query and ranks the documents.
- **Top-k** means the first `k` documents in that ranking. This competition uses `k=5`.

The public files are:

| File | Purpose |
|---|---|
| `documents.csv` | Searchable agricultural documents |
| `train_queries.csv` | Training queries and their positive-document summary |
| `qrels_train.csv` | Graded relevance labels for local evaluation |
| `test_queries.csv` | Queries for the Kaggle submission |
| `sample_submission.csv` | Required submission columns and example layout |

The final file has exactly two columns: `QueryId,DocumentId`. **There is no score or relevance column.** Within each query, the first row is rank 1, the second is rank 2, and so on.

In [ ]:
# Standard libraries
from pathlib import Path
import math
import numpy as np
import pandas as pd

# Kaggle mounts competition data under this directory.
base = Path('/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers')

# This fallback helps if Kaggle displays the dataset under a slightly different slug.
if not (base / 'documents.csv').exists():
    candidates = list(Path('/kaggle/input').glob('**/documents.csv'))
    if not candidates:
        raise FileNotFoundError(
            "documents.csv was not found. Open the notebook's Input panel and add "
            "the competition data, then run this cell again."
        )
    base = candidates[0].parent

print('Reading competition files from:', base)
documents = pd.read_csv(base / 'documents.csv')
train_queries = pd.read_csv(base / 'train_queries.csv')
qrels = pd.read_csv(base / 'qrels_train.csv')
test_queries = pd.read_csv(base / 'test_queries.csv')
sample_submission = pd.read_csv(base / 'sample_submission.csv')

print('documents:', documents.shape)
print('train queries:', train_queries.shape)
print('qrels:', qrels.shape)
print('test queries:', test_queries.shape)
display(documents.head(2))
display(train_queries.head(2))
display(qrels.head(2))
display(sample_submission.head(10))

Reading competition files from: /kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers
documents: (695, 9)
train queries: (308, 3)
qrels: (4194, 3)
test queries: (200, 2)


,document_id,title,text,source,crop,country,origin,source_url,license
0,1,Drought and erratic rainfall: the risk to crops,Drought and erratic rainfall and its impact on...,FAO,(general),Kenya,synthetic,NaN,synthetic (CC0)
1,2,Adapting to drought and erratic rainfall (Guin...,Adapting to drought and erratic rainfall in th...,IITA,(general),Tanzania,synthetic,NaN,synthetic (CC0)


,query_id,query,positive_docs
0,1,How do I cope with flooding and excess rain on...,8 9 10 7 6
1,2,How can I adapt my farming to flooding and exc...,8 9 10 7 6


,query_id,document_id,relevance
0,1,8,3.0
1,1,9,3.0


,QueryId,DocumentId
0,1001,1
1,1001,2
2,1001,3
3,1001,4
4,1001,5
5,1002,1
6,1002,2
7,1002,3
8,1002,4
9,1002,5


## 2. Check and explore the data

Checking schemas early produces clearer errors than discovering a missing column at submission time. We also verify that identifiers are unique where they should be.

In [ ]:
required_columns = {
    'documents': {'document_id', 'title', 'text'},
    'train_queries': {'query_id', 'query'},
    'qrels': {'query_id', 'document_id', 'relevance'},
    'test_queries': {'query_id', 'query'},
}
frames = {
    'documents': documents,
    'train_queries': train_queries,
    'qrels': qrels,
    'test_queries': test_queries,
}
for name, expected in required_columns.items():
    missing = expected - set(frames[name].columns)
    assert not missing, f'{name} is missing columns: {sorted(missing)}'

assert documents['document_id'].is_unique, 'Each document_id must be unique.'
assert train_queries['query_id'].is_unique, 'Each training query_id must be unique.'
assert test_queries['query_id'].is_unique, 'Each test query_id must be unique.'
assert qrels['relevance'].between(0, 3).all(), 'Relevance values must be between 0 and 3.'

# Inspect one training query and the documents judged relevant for it.
example_query = train_queries.iloc[0]
example_qrels = qrels[
    (qrels['query_id'] == example_query['query_id']) & (qrels['relevance'] > 0)
].sort_values('relevance', ascending=False)
example_docs = example_qrels.merge(documents, on='document_id', how='left')

print('Query:', example_query['query'])
display(example_docs[['document_id', 'relevance', 'title', 'text']].head())

Query: How do I cope with flooding and excess rain on my farm?


,document_id,relevance,title,text
0,8,3.0,Adapting to flooding and excess rain (Highlands),Adapting to flooding and excess rain in the co...
1,9,3.0,Adapting to flooding and excess rain (Humid fo...,Adapting to flooding and excess rain in the hu...
2,10,3.0,Adapting to flooding and excess rain (Sahel),Adapting to flooding and excess rain in the ho...
3,7,3.0,Adapting to flooding and excess rain (Guinea s...,Adapting to flooding and excess rain in the Gu...
4,6,2.0,Flooding and excess rain: the risk to crops,Flooding and excess rain and its impact on far...


In [ ]:
#===========================================
# DATASET INSPECTION & METRICS SUMMARY
#==========================================
# Import required analysis tools
import pandas as pd
import numpy as nd

#1. Dataset Dimensions & Basics
print("=== DATASET SHAPES ===")
print(f"Total Documents:     {len(documents):,}")
print(f"Total Train Queries:  {len(train_queries):,}")
print(f"Total Text Queries:   {len(train_queries):,}")
print(f"Total Qrels Labels:   {len(qrels):,}")

#2. Check for the missing valuesin text columns
print("\n=== MISSING VALUES ===")
print("Documents missing titles:", documents['title'].isna().sum())
print("Documents missing text:  ", documents['text'].isna().sum())
print("Train queries missing:  ", train_queries['query'].isna().sum())
print("Test queries missing:    ", test_queries['query'].isna().sum())

#3. Text Large Characteristics (Word Counts)
documents['title_word_count'] = documents['title'].fillna('').apply(lambda x: len(x.split()))
documents['text_word_count'] = documents['text'].fillna('').apply(lambda x: len(x.split()))
train_queries['query_word_count'] = train_queries['query'].fillna('').apply(lambda x: len(x.split()))

print("\n=== SUMMARY STATISTICS ===")
print("Document Text Length (words):")
print(documents['text_word_count'].describe().round(1))

print("\nTrain Query Length (word):")
print(train_queries['query_word_count'].describe().round(1))

#4. Relevance Label Distribution
print("\n=== RELEVANCE VALUE COUNTS ===")
print(qrels['relevance'].value_counts().sort_index())


=== DATASET SHAPES ===
Total Documents:     695
Total Train Queries:  308
Total Text Queries:   308
Total Qrels Labels:   4,194

=== MISSING VALUES ===
Documents missing titles: 0
Documents missing text:   0
Train queries missing:   0
Test queries missing:     0

=== SUMMARY STATISTICS ===
Document Text Length (words):
count    695.0
mean      66.1
std       28.7
min       20.0
25%       57.0
50%       65.0
75%       73.0
max      180.0
Name: text_word_count, dtype: float64

Train Query Length (word):
count    308.0
mean       9.4
std        2.0
min        6.0
25%        8.0
50%        9.0
75%       11.0
max       16.0
Name: query_word_count, dtype: float64

=== RELEVANCE VALUE COUNTS ===
relevance
0.0    2724
1.0     728
2.0     186
3.0     556
Name: count, dtype: int64


## 3. Competition metric: nDCG@5

**Discounted Cumulative Gain (DCG)** rewards relevant documents, but discounts documents that appear lower in the ranking:

$$DCG@5 = \sum_{i=1}^{5} \frac{rel_i}{\log_2(i+1)}$$

**Ideal DCG (IDCG)** is the DCG of the best possible ordering. `nDCG = DCG / IDCG`, so the result is between 0 and 1. A relevant document at rank 1 helps more than the same document at rank 5.

The following local evaluator mirrors that definition. It is for learning and validation; Kaggle computes the official score on hidden test labels.

In [ ]:
def dcg(relevances, k=5):
    """Return discounted cumulative gain for an ordered relevance list."""
    values = np.asarray(list(relevances)[:k], dtype=float)
    if len(values) == 0:
        return 0.0
    discounts = np.log2(np.arange(2, len(values) + 2))
    return float(np.sum(values / discounts))


def evaluate_ndcg_at_5(predictions, qrels_frame=qrels):
    """Evaluate a {query_id: [ranked document_ids]} prediction dictionary."""
    relevance_lookup = {
        query_id: dict(zip(group['document_id'], group['relevance']))
        for query_id, group in qrels_frame.groupby('query_id')
    }
    per_query = {}
    for query_id, judged_docs in relevance_lookup.items():
        ranked_docs = predictions.get(query_id, [])[:5]
        predicted_relevance = [judged_docs.get(doc_id, 0) for doc_id in ranked_docs]
        ideal_relevance = sorted(judged_docs.values(), reverse=True)[:5]
        ideal_dcg = dcg(ideal_relevance, k=5)
        per_query[query_id] = dcg(predicted_relevance, k=5) / ideal_dcg if ideal_dcg else 0.0
    return float(np.mean(list(per_query.values()))), per_query


# A quick sanity check: an ideal ranking should score 1.0 for this toy query.
toy_qrels = pd.DataFrame({
    'query_id': ['q1', 'q1', 'q1'],
    'document_id': ['d1', 'd2', 'd3'],
    'relevance': [3, 2, 1],
})
toy_score, _ = evaluate_ndcg_at_5({'q1': ['d1', 'd2', 'd3']}, toy_qrels)
assert np.isclose(toy_score, 1.0)
print('Metric sanity check passed.')

Metric sanity check passed.


## 4. Build a TF-IDF retrieval baseline

TF-IDF gives high weight to words that are important in one document but uncommon across the corpus. We combine each title and body, learn document vectors, turn queries into vectors in the same space, and use cosine similarity to rank documents.

This is an **unsupervised** baseline: it does not fit relevance labels or tune hyperparameters on them. Evaluating all training queries is therefore a useful baseline check. If you later learn weights or tune settings using qrels, create a held-out validation set. Related or paraphrased queries should remain in the same fold to avoid leakage; the `positive_docs` field can help identify queries sharing the same relevant-document family.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

# Titles often contain compact topic information, so include both title and body.
document_text = (
    documents['title'].fillna('').astype(str)
    + '. '
    + documents['text'].fillna('').astype(str)
)

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words='english',
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
)
document_matrix = normalize(vectorizer.fit_transform(document_text))
document_ids = documents['document_id'].to_numpy()

print('Document matrix shape:', document_matrix.shape)
print('Vocabulary size:', len(vectorizer.vocabulary_))

Document matrix shape: (695, 4094)
Vocabulary size: 4094


In [ ]:
def retrieve_tfidf(query_frame, k=5):
    """Return top-k document IDs for every query in query_frame."""
    query_matrix = normalize(vectorizer.transform(query_frame['query'].fillna('').astype(str)))
    similarities = query_matrix @ document_matrix.T
    results = {}
    for row_number, query_id in enumerate(query_frame['query_id']):
        scores = similarities.getrow(row_number).toarray().ravel()
        # argpartition finds the candidates efficiently; argsort then gives exact rank order.
        candidate_positions = np.argpartition(-scores, min(k, len(scores)) - 1)[:k]
        ranked_positions = candidate_positions[np.argsort(-scores[candidate_positions])]
        results[query_id] = document_ids[ranked_positions].tolist()
    return results


train_predictions = retrieve_tfidf(train_queries, k=5)
tfidf_score, query_scores = evaluate_ndcg_at_5(train_predictions)
print(f'TF-IDF training-query nDCG@5: {tfidf_score:.4f}')

# Inspect the hardest queries to guide improvements.
score_table = train_queries[['query_id', 'query']].copy()
score_table['ndcg_at_5'] = score_table['query_id'].map(query_scores)
display(score_table.sort_values('ndcg_at_5').head(10))

TF-IDF training-query nDCG@5: 0.5131


,query_id,query,ndcg_at_5
307,308,What should I do about an outbreak of soil aci...,0.0
306,307,How do I manage soil acidification in maize?,0.0
20,21,How do I manage blast in rice?,0.0
21,22,What should I do about an outbreak of blast in...,0.0
30,31,How do I manage rice blast in rice?,0.0
280,281,What damage does weevil severity do to plantain?,0.0
215,216,Is it sulphur deficiency or another deficiency...,0.0
193,194,What fertiliser corrects potassium deficiency ...,0.0
133,134,What fertiliser corrects phosphorus deficiency...,0.0
132,133,How do I fix phosphorus deficiency in maize?,0.0


In [ ]:
#=========================================================
#EXPERIMENT TRACKING TABLE & BASELINE RECORD
#==========================================================
#we create a structured pandas DataFrame to track every experiment.
import pandas as pd

#Define experiment log structure
experiments = []

def log_experiment(name, method,parameters, ndcg_score, notes=""):
    """Appends experiment details to our tracking ledger."""
    experiments.append({
        "Experiment": name,
        "Method": method,
        "Parameters": parameters,
        "nDCG@5": round(ndcg_score, 4),
        "Notes": notes
    })
    return pd.DataFrame(experiments)

# Record the initial TF-IDF Baseline score evaluated in Section 4
exp_df = log_experiment(
    name = "Baseline",
    method = "TF-IDF (Title + Text)",
    parameters = "ngram =(1,2), min_df = 2, sublinear_tf = True",
    ndcg_score = tfidf_score, #Used tfidf_score computed in Section 4 cell
    notes = "Official Starter Baseline"

)
display(exp_df)

,Experiment,Method,Parameters,nDCG@5,Notes
0,Baseline,TF-IDF (Title + Text),"ngram =(1,2), min_df = 2, sublinear_tf = True",0.5131,Official Starter Baseline


## 5. Create and validate `submission.csv`

Kaggle reads row order as rank order. The code below adds five rows per test query, from the best-ranked document to the fifth-ranked document. Do not sort these rows afterward.

In [ ]:
def predictions_to_submission(predictions, query_frame, valid_document_ids, k=5):
    """Convert ranked predictions to the exact competition submission format."""
    rows = []
    for query_id in query_frame['query_id']:
        ranked_docs = predictions.get(query_id, [])
        assert len(ranked_docs) == k, f'{query_id} has {len(ranked_docs)} predictions, expected {k}.'
        for document_id in ranked_docs:
            rows.append({'QueryId': query_id, 'DocumentId': document_id})

    submission = pd.DataFrame(rows, columns=['QueryId', 'DocumentId'])
    expected_query_order = query_frame['query_id'].repeat(k).tolist()
    assert list(submission.columns) == ['QueryId', 'DocumentId']
    assert len(submission) == len(query_frame) * k
    assert submission['QueryId'].tolist() == expected_query_order, 'Query/rank order changed.'
    assert submission.groupby('QueryId', sort=False).size().eq(k).all()
    assert not submission.duplicated(['QueryId', 'DocumentId']).any(), 'A query contains duplicate documents.'
    assert set(submission['DocumentId']).issubset(set(valid_document_ids)), 'Unknown document ID found.'
    return submission


test_predictions = retrieve_tfidf(test_queries, k=5)
submission = predictions_to_submission(
    test_predictions,
    test_queries,
    valid_document_ids=documents['document_id'],
    k=5,
)

output_path = Path('/kaggle/working/submission.csv')
submission.to_csv(output_path, index=False)
print('Saved:', output_path)
print('Shape:', submission.shape)
display(submission.head(10))

Saved: /kaggle/working/submission.csv
Shape: (1000, 2)


,QueryId,DocumentId
0,1001,1
1,1001,2
2,1001,4
3,1001,3
4,1001,5
5,1002,1
6,1002,2
7,1002,4
8,1002,3
9,1002,5


## 6. Submit from your committed Kaggle Notebook

1. Confirm the latest cell wrote **`submission.csv`** to `/kaggle/working/`.
2. Check that the file contains exactly **`QueryId,DocumentId`**, **1,000 rows** (200 queries × 5), exactly 5 rows per query, and no duplicate `(QueryId, DocumentId)` pairs.
3. Click **Save Version**, choose **Save & Run All**, and wait for the committed run to finish successfully. An interactive draft is not a valid final notebook.
4. Open the saved version's **Output** panel, confirm `submission.csv` is present, and select **Submit to Competition**. If that button is unavailable, download the output CSV and upload it on the competition submission page, then provide the committed notebook link as required by the rules.
5. Keep the submitted notebook private during the competition unless organizers explicitly request publication.

The notebook must be reproducible from top to bottom and must not read the private solution file or hard-code hidden test labels.

In [ ]:
import math
from collections import Counter
import pandas as pd
documents_df = pd.read_csv("/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/documents.csv")
import re
import numpy as np

def hybrid_retrieve(query, query_id, k = 5, alpha = 60):
    #1. Get BM25top 20 canidates
    bm25_ids = retrieve_bm25(query, k = 20)

    #2. Get Dense top 20 candidates
    dense_ids = []
    if 'dense_test_prediction' in globals() and query_id in dense_test_prediction:
        dense_ids = dense_test_prediction[query_id][:20]

    #3. Calculate Reciprocal Rank Fusion (RRF) scores
    rrf_scores = {}

    for rank, doc_id in enumerate(bm25_ids):
        rrf_scores[doc_id] = rrf_scores.get(doc_id,0) +(1.0 / (alpha + rank + 1))

    for rank, doc_id in enumerate(dense_ids):
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + (1.0 / (alpha + rank + 1))

    # 4. Sort documents by combine RRF score
    sorted_docs = sorted(rrf_scores.keys(), key =lambda x: rrf_scores[x],reverse = True)
    return sorted_docs[:k]

#Function to clean and tokenize text
def clean_tokenize(text):
    #Remove punctuation marks and convert to lowercase
    cleaned_text = re.sub(r'[^\w\s]', '', str(text)).lower()
    return cleaned_text.split()

#Build index with cleaned text
corpus = [
    clean_tokenize(f"{doc['title']} {doc['text']}")
    for doc in documents_df.to_dict("records")
]




class PureBM25:

    def __init__(self, corpus, k1 = 1.5, b = 0.75):
        self.k1 = k1
        self.b = b
        self.corpus = corpus
        self.doc_len = [len(doc) for doc in corpus]
        self.avgdl = sum(self.doc_len) / len(corpus) if corpus else 0
        self.doc_freqs = []
        self.nd = len(corpus)
        self.df = Counter()

        for doc in corpus:
            frequencies = Counter(doc)
            self.doc_freqs.append(frequencies)
            for word in frequencies:
              self.df[word] += 1

    def get_scores(self, query):
      scores = [0.0] * self.nd
      for word in query:
        if word not in self.df:
          continue
        df = self.df[word]
        # Standard Lucene/BM25 IDF formula
        idf = math.log(1 + (self.nd - df + 0.5) /df + 0.5)

        for idx, doc_freqs in enumerate(self.doc_freqs):
            freq = doc_freqs[word]
            if freq > 0:
              denom = freq + self.k1 * (
                  1 - self.b + self.b * (self.doc_len[idx] / self.avgdl))
              scores[idx] += idf * (freq * (self.k1 + 1)) /denom
      return scores

#Weight document titles 2 * heavily during BM25 corpus construction
corpus = [
    clean_tokenize(f"{doc['title']} {doc['title']} {doc['text']}")
    for doc in documents_df.to_dict("records")
]


bm25 = PureBM25(corpus)

def retrieve_bm25(query, k = 5):
    tokenized_query = clean_tokenize(query)
    scores = bm25.get_scores(tokenized_query)
    top_indices = sorted(range(len(scores)), key =lambda i: scores[i], reverse = True)[:k]
    return documents_df.iloc[top_indices]["document_id"].tolist()

In [ ]:
sample_query = "maize pest management"
results = retrieve_bm25(sample_query, k = 5)
print("Top 5 Document IDs:", results)

Top 5 Document IDs: [650, 646, 633, 651, 647]


## 7. Optional: dense retrieval with a model attached from Kaggle Models

Dense retrieval can match concepts even when the query and document use different words. To use it without internet downloads:

1. In the notebook's right-hand **Input** panel, choose **Add Input**.
2. Open **Models** and search for a sentence-embedding model such as `all-MiniLM-L6-v2`.
3. Attach a Transformers/PyTorch version containing `config.json`, tokenizer files, and model weights.
4. A GPU is optional for this small corpus, but you can enable one in notebook settings.
5. Run the cells below. If auto-detection finds the wrong model, set `MODEL_PATH` manually to its folder under `/kaggle/input`.

The example uses Kaggle's installed `transformers` and `torch` libraries. It does not call the internet or alter the competition data.

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer

# Find attached Hugging Face-style model folders.
model_candidates = [
    path.parent
    for path in Path('/kaggle/input').glob('**/config.json')
    if any(path.parent.glob('tokenizer*'))
    and (any(path.parent.glob('*.safetensors')) or any(path.parent.glob('pytorch_model*.bin')))
]
if not model_candidates:
    raise FileNotFoundError(
        'No attached transformer model was found. Use Add Input → Models, attach '
        'a sentence-embedding model, and run this cell again.'
    )

MODEL_PATH = model_candidates[0]  # Replace this if several models are attached.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
encoder = AutoModel.from_pretrained(MODEL_PATH, local_files_only=True).to(device)
encoder.eval()
print('Using model:', MODEL_PATH)
print('Device:', device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Using model: /kaggle/input/models/weedoo/e5-base-v2/transformers/01/1
Device: cpu


In [ ]:
def mean_pool(last_hidden_state, attention_mask):
    """Average token embeddings while ignoring padding tokens."""
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts


def encode_texts(texts, batch_size=32, max_length=256):
    """Encode text in batches and return L2-normalized NumPy vectors."""
    batches = []
    texts = list(pd.Series(texts).fillna('').astype(str))
    with torch.no_grad():
        for start in range(0, len(texts), batch_size):
            tokens = tokenizer(
                texts[start:start + batch_size],
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors='pt',
            ).to(device)
            output = encoder(**tokens)
            embeddings = mean_pool(output.last_hidden_state, tokens['attention_mask'])
            embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)
            batches.append(embeddings.cpu().numpy())
    return np.vstack(batches)


dense_document_vectors = encode_texts(document_text, batch_size=32)
print('Dense document matrix shape:', dense_document_vectors.shape)

Dense document matrix shape: (695, 768)


In [ ]:
def retrieve_dense(query_frame, k=5):
    query_vectors = encode_texts(query_frame['query'], batch_size=32)
    similarities = query_vectors @ dense_document_vectors.T
    results = {}
    for row_number, query_id in enumerate(query_frame['query_id']):
        scores = similarities[row_number]
        candidates = np.argpartition(-scores, min(k, len(scores)) - 1)[:k]
        ranked = candidates[np.argsort(-scores[candidates])]
        results[query_id] = document_ids[ranked].tolist()
    return results


dense_train_predictions = retrieve_dense(train_queries, k=5)
dense_score, _ = evaluate_ndcg_at_5(dense_train_predictions)
print(f'Dense training-query nDCG@5: {dense_score:.4f}')

# This intentionally replaces the earlier baseline submission only after validation.
dense_test_predictions = retrieve_dense(test_queries, k=5)
dense_submission = predictions_to_submission(
    dense_test_predictions,
    test_queries,
    valid_document_ids=documents['document_id'],
    k=5,
)
dense_submission.to_csv('/kaggle/working/submission.csv', index=False)
print('Dense submission saved to /kaggle/working/submission.csv')
display(dense_submission.head(10))

Dense training-query nDCG@5: 0.7856
Dense submission saved to /kaggle/working/submission.csv


,QueryId,DocumentId
0,1001,3
1,1001,4
2,1001,5
3,1001,2
4,1001,8
5,1002,4
6,1002,5
7,1002,3
8,1002,2
9,1002,10


In [ ]:
import os
import numpy as np
import torch
from transformers import AutoModel, AutoTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else"cpu")

#1. Print the exact directories in /kaggle/input to cofirm folder names
print("Available input directories:", os.listdir("/kaggle/input"))

#2. Define Local Paths
model_path = "/kaggle/input/models/srg9000/all-minilm-l6-v2"
print("Files inside Minilm:")
for root,dirs,files in os.walk(model_path):
    for file in files:
        print(os.path.join(root, file))

print("Contents of /kaggle/input:")
for root, dirs,files in os.walk("/kaggle/input"):
    level = root.replace("/kaggle/input", "").count(os.sep)
    if level <= 4:
        print(root)

def find_model_directory(base_path):
    """
    find the actual Hugging Face model directory
    containing config.json and tokenizer files.
    """
    for root, dirs, files in os.walk(base_path):
        if "config.json" in files:
            return root

    raise FileNotFoundError(
        f"Could not find a valid Hugging Face model directory inside: {base_path}"
    )
#Parent directories provided by kaggle
MODEL_BASE_PATHS = {
    "minilm": "/kaggle/input/models/srg9000/all-minilm-l6-v2",
    "baai": "/kaggle/input/models/jonathanchan/baai",
    "e5": "/kaggle/input/models/weedoo/e5-base-v2"
}
#Automatically locate the real model directories
PATHS = {}
for name, path in MODEL_BASE_PATHS.items():
    PATHS[name] = find_model_directory(path)
    print(f"{name}:")
    print(f"  Path: {PATHS[name]}")
    print(f"  Exists: {os.path.exists(PATHS[name])}")
    print()
#3. Embedding Generator Function
def get_model_embeddings(model_path, texts, prefix ="", batch_size = 32):
    # Load tokenizer and model strictly from local files
    tokenizer = AutoTokenizer.from_pretrained(
        model_path, local_files_only = True
    )
    model = AutoModel.from_pretrained(model_path, local_files_only = True).to(
        device
    )
    model.eval()

    formatted_texts = [f"{prefix}{t}" for t in texts]
    all_embeddings = []

    for i in range(0, len(formatted_texts), batch_size):
        batch = formatted_texts[i:i + batch_size]

        #FIX: Ensure return_tensors =  'pt' is set
        encoded = tokenizer(
            batch,
            padding = True,
            truncation = True,
            max_length = 512,
            return_tensors = "pt"
        ).to(device)

        outputs = model(**encoded)
        mask = encoded["attention_mask"][..., None]

        #Mean pooling calculation
        sum_embeddings = torch.sum(outputs.last_hidden_state * mask, dim = 1)
        sum_mask = torch.clamp(mask.sum(dim=1), min=1e-9)
        embedding = sum_embeddings / sum_mask

        #Normalize embeddings
        embedding = torch.nn.functional.normalize(embedding,p = 2,dim = 1)

        all_embeddings.append(embedding.detach().cpu().numpy())

    return np.vstack(all_embeddings)

#4. Extract Document Texts
raw_docs = [
    f"{doc['title']}{doc['text']}"
    for doc in documents_df.to_dict("records")
]

# 5 .Generate Embeddings for all 3 models
print("Generate MiniLM Embeddings...")
docs_minilm = get_model_embeddings(PATHS["minilm"], raw_docs)

print("Generate BAAI Embeddings...")
docs_baai = get_model_embeddings(PATHS["baai"], raw_docs)

print("Gnerate E5 Embeddings...")
docs_e5 = get_model_embeddings(PATHS["e5"],raw_docs, prefix = "passage: ")

def retrieve_dense_ensemble(query, k = 5):
    """
    Retrieve documents using an ensemble of MiniLM, BAAI, and E5

    Each model indepently scores the query against its own
    document enbeddings. The three similarity scores are
    then averaged to produce the final ranking.
    """

    #Generate the query embedding for each model
    query_minilm = get_model_embeddings(
        PATHS["minilm"],
        [query]
    )

    query_baai = get_model_embeddings(
        PATHS["baai"],
        [query]
    )

    #E5 uses a diffenrent prefix for querys
    query_e5 = get_model_embeddings(
        PATHS["e5"],
        [query],
        prefix = "query: "
    )

    #Calculate similarity for each model
    scores_minilm = query_minilm @ docs_minilm.T
    scores_baai = query_baai @ docs_baai.T
    scores_e5 = query_e5 @ docs_e5.T

    # Combine the three model scores
    ensemble_scores = (
        scores_minilm[0]
        + scores_baai[0]
        + scores_e5[0]
    ) / 3.0

    #Get the top-k document
    k = min(k, len(ensemble_scores))

    candidates = np.argpartition(
        -ensemble_scores,
        k - 1
    )[:k]

    ranked = candidates[
        np.argsort(-ensemble_scores[candidates])
    ]

    return document_ids[ranked].tolist()

Available input directories: ['competitions', 'models']
Files inside Minilm:
/kaggle/input/models/srg9000/all-minilm-l6-v2/transformers/default/1/all-MiniLM-L6-v2/rust_model.ot
/kaggle/input/models/srg9000/all-minilm-l6-v2/transformers/default/1/all-MiniLM-L6-v2/config.json
/kaggle/input/models/srg9000/all-minilm-l6-v2/transformers/default/1/all-MiniLM-L6-v2/README.md
/kaggle/input/models/srg9000/all-minilm-l6-v2/transformers/default/1/all-MiniLM-L6-v2/tokenizer.json
/kaggle/input/models/srg9000/all-minilm-l6-v2/transformers/default/1/all-MiniLM-L6-v2/tf_model.h5
/kaggle/input/models/srg9000/all-minilm-l6-v2/transformers/default/1/all-MiniLM-L6-v2/data_config.json
/kaggle/input/models/srg9000/all-minilm-l6-v2/transformers/default/1/all-MiniLM-L6-v2/train_script.py
/kaggle/input/models/srg9000/all-minilm-l6-v2/transformers/default/1/all-MiniLM-L6-v2/tokenizer_config.json
/kaggle/input/models/srg9000/all-minilm-l6-v2/transformers/default/1/all-MiniLM-L6-v2/sentence_bert_config.json
/kagg

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: /kaggle/input/models/srg9000/all-minilm-l6-v2/transformers/default/1/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generate BAAI Embeddings...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: /kaggle/input/models/jonathanchan/baai/transformers/bge-small-en-v1.5/1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Gnerate E5 Embeddings...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
#Test a sample query
sample_query_text = "How to manage stemborers in maize framing?"
sample_query_id = 0 # Replace with a valid query_id from your dataset

retrieved_docs = hybrid_retrieve(sample_query_text, query_id = sample_query_id, k = 5)

print("Sample Query:", sample_query_text)
print("Top 5 Retrieved Document IDs:", retrieved_docs)

Sample Query: How to manage stemborers in maize framing?
Top 5 Retrieved Document IDs: [117, 109, 93, 125, 529]


## 8. Ideas for improvement

- **Error analysis:** read low-scoring training queries and compare the retrieved documents with their qrels.
- **Title weighting:** repeat the title or build separate title/body similarities with a tuned weight.
- **Hybrid retrieval:** combine TF-IDF and dense rankings using reciprocal-rank fusion rather than comparing incomparable raw scores.
- **Query expansion:** add carefully chosen agricultural synonyms or crop names.
- **Reranking:** retrieve a larger candidate set, then use a stronger attached model to reorder only those candidates.
- **Leakage-safe validation:** when tuning, keep queries that share the same positive-document set in one fold. A family key can be made by sorting the IDs in `positive_docs`.

Change one thing at a time and record the validation score. A trustworthy experiment table is more useful than many undocumented attempts.

In [ ]:
def retrieve_hybrid(query, k = 5, alpha = 60):
    bm25_ids = retrieve_bm25(query, k = 20)
    dense_ids = retrieve_dense_ensemble(query, k = 20)

    rrf_scores = {}
    for rank, doc_id in enumerate(bm25_ids):
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + (1.0 / (alpha + rank + 1))

    for rank, doc_id in enumerate(dense_ids):
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + (1.0 / (alpha + rank + 1))

    sorted_docs = sorted(rrf_scores.keys(), key = lambda x: rrf_scores[x], reverse = True)
    return sorted_docs[:k]

In [ ]:
import os
import pandas as pd

competition_path = "/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers"

print("Files available in the competition folder:")

for file in os.listdir(competition_path):
    print(file)

#Load test queries
test_df = pd.read_csv(
    "/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/test_queries.csv"
)
print("Test queries loaded successfully!")
print(test_df.head())
print("Number of test queries:", len(test_df))
#Format retrieved document IDs as space-separated string
test_df["retrieved_doc_ids"] = test_df["query"].apply(lambda q: " ".join(map(str,retrieve_hybrid(q, k = 5))))

submission_df = test_df[["query_id", "retrieved_doc_ids"]] if "query_id" in test_df.columns else test_df
submission_df.to_csv("submission.csv", index = False)

print("Ensemble submission.csv generated successfully!")
print(submission_df.head())

submission = pd.read_csv('/kaggle/working/submission.csv')

print("Shape:", submission.shape)
print("Columns", submission.columns.tolist())
print("\nNumber of test queries:", submission["query_id"].nunique())

print("\nDocuments retrieved per query:")
print(
    submission["retrieved_doc_ids"]
    .str.split()
    .str.len()
    .value_counts()
)
print("\nFirst 10 rows:")
display(submission.head(10))

# Convert ensemble results to the required kaggle submission format
final_submission = predictions_to_submission(
    dense_test_predictions,
    test_queries,
    valid_document_ids = documents["document_id"],
    k = 5,
)
final_submission.to_csv(
    "/kaggle/working/final_submission.csv",
    index = False
)

print("Final Kaggle submission created!")
print("Shape:", final_submission.shape)
print("Columns:", final_submission.columns.tolist())

display(final_submission.head(10))

Files available in the competition folder:
test_queries.csv
sample_submission.csv
documents.csv
dataset-metadata.json
train_queries.csv
qrels_train.csv
baseline_submission.csv
Test queries loaded successfully!
   query_id                                              query
0      1001  How do I cope with drought and erratic rainfal...
1      1002  How can I adapt my farming to drought and erra...
2      1003  How does drought and erratic rainfall affect m...
3      1004  What is the risk of drought and erratic rainfa...
4      1005         How do I cope with heat stress on my farm?
Number of test queries: 200


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: /kaggle/input/models/srg9000/all-minilm-l6-v2/transformers/default/1/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: /kaggle/input/models/jonathanchan/baai/transformers/bge-small-en-v1.5/1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Ensemble submission.csv generated successfully!
   query_id retrieved_doc_ids
0      1001         1 4 5 3 2
1      1002         1 5 4 3 2
2      1003         1 5 3 4 2
3      1004        1 5 4 16 3
4      1005    11 14 13 12 15
Shape: (200, 2)
Columns ['query_id', 'retrieved_doc_ids']

Number of test queries: 200

Documents retrieved per query:
retrieved_doc_ids
5    200
Name: count, dtype: int64

First 10 rows:


,query_id,retrieved_doc_ids
0,1001,1 4 5 3 2
1,1002,1 5 4 3 2
2,1003,1 5 3 4 2
3,1004,1 5 4 16 3
4,1005,11 14 13 12 15
5,1006,11 13 12 14 15
6,1007,11 13 15 12 14
7,1008,11 14 13 15 12
8,1009,16 19 18 20 17
9,1010,16 18 20 17 19


Final Kaggle submission created!
Shape: (1000, 2)
Columns: ['QueryId', 'DocumentId']


,QueryId,DocumentId
0,1001,3
1,1001,4
2,1001,5
3,1001,2
4,1001,8
5,1002,4
6,1002,5
7,1002,3
8,1002,2
9,1002,10


## 9. Troubleshooting and responsible competition practice

| Problem | What to check |
|---|---|
| `documents.csv was not found` | Join the competition and add its data through the notebook Input panel. |
| Model not found | Attach a Kaggle Model containing Hugging Face config, tokenizer, and weight files. |
| Notebook runs out of memory | Reduce `batch_size` or `max_length`; restart the session after a crash. |
| Submission has the wrong shape | It must contain exactly five rows per test query and only `QueryId,DocumentId`. |
| Score is unexpectedly poor | Confirm that rows within each query are best-to-worst and document IDs are valid. |
| Submit button is unavailable | Use Save Version → Save & Run All and submit from that completed version's Output. |

Do not use hidden solution files, manually encode test answers, or copy another participant's private work. Your method should learn only from the provided public competition data and permitted pretrained models. Explain your approach, experiments, and limitations in your notebook.

In [ ]:
test_result = retrieve_dense_ensemble(
    "How do I cope with drought and erratic rainfall?",
    k = 5
)

print(test_result)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: /kaggle/input/models/srg9000/all-minilm-l6-v2/transformers/default/1/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: /kaggle/input/models/jonathanchan/baai/transformers/bge-small-en-v1.5/1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[1, 4, 2, 5, 3]
